# Nexa - ASR benchmark (Member 2)

Runs the Whisper candidates over the team's labelled clips on a free Colab GPU.

**Before you run**
1. `Runtime -> Change runtime type -> T4 GPU`.
2. Put this in Google Drive at `MyDrive/Nexa/`:
   - `src/` and `scripts/` (copy from the repo)
   - `audio/` = the dataset: `audio/<category>/<clip>.wav` + `<clip>.txt` (+ optional `<clip>.json`)
3. `Runtime -> Run all`.

Results land in `MyDrive/Nexa/benchmark_out/` (`summary.md`, `results.csv`).

In [ ]:
!nvidia-smi -L
!pip install -q faster-whisper psutil

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/Nexa'
DATASET = f'{ROOT}/audio'
OUT = f'{ROOT}/benchmark_out'

# What to compare. Add a CTranslate2-converted Egyptian fine-tune path/repo here later.
MODELS = ['whisper-small', 'whisper-medium']
# auto = Whisper detects language once per file; ar = forced Arabic. Comparing both matters for code-switching.
LANGUAGES = ['auto', 'ar']
# Optional: a mixed-language sentence that nudges Whisper to keep English words in Latin script.
PROMPT = ''

import os
assert os.path.isdir(f'{ROOT}/src/nexa'), 'Copy the repo src/ and scripts/ into MyDrive/Nexa/'
assert os.path.isdir(DATASET), 'Copy the dataset into MyDrive/Nexa/audio/'

In [ ]:
prompt_arg = f'--prompt "{PROMPT}"' if PROMPT else ''
!cd {ROOT} && python scripts/benchmark_asr.py --dataset audio --models {' '.join(MODELS)} --languages {' '.join(LANGUAGES)} {prompt_arg} --out benchmark_out --limit 3

The cell above is a **smoke test on 3 clips**. If it works, run the full set below (remove `--limit`).

In [ ]:
!cd {ROOT} && python scripts/benchmark_asr.py --dataset audio --models {' '.join(MODELS)} --languages {' '.join(LANGUAGES)} {prompt_arg} --out benchmark_out

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f'{OUT}/summary.md', encoding='utf-8').read()))

## Where does each model fail?
Worst clips per model - read the hypotheses yourself; numbers alone will not tell you *why*
(transliterated English, dropped dates, hallucinated text on noisy clips...).

In [ ]:
import pandas as pd
df = pd.read_csv(f'{OUT}/results.csv', encoding='utf-8-sig')
df['wer'] = df.word_errors / df.ref_words.clip(lower=1)
worst = df.sort_values('wer', ascending=False).groupby('model').head(5)
pd.set_option('display.max_colwidth', 120)
worst[['model', 'clip', 'wer', 'hypothesis']]

## Optional: convert a Hugging Face Whisper fine-tune for faster-whisper
faster-whisper only loads CTranslate2 models. Set `HF_REPO` to the Egyptian code-switch checkpoint the team picks
(nobody has verified a specific one yet), run this cell, then add the output folder to `MODELS`.

In [ ]:
HF_REPO = ''   # e.g. 'org/whisper-egyptian-codeswitch'
if HF_REPO:
    !pip install -q ctranslate2 transformers[torch]
    !ct2-transformers-converter --model {HF_REPO} --output_dir {ROOT}/models/whisper/custom --quantization float16 --copy_files tokenizer.json preprocessor_config.json
    print('Add to MODELS:', f'{ROOT}/models/whisper/custom')